In [1]:
pip install uv

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# import adapters.
import regex as re
import heapq

special_tokens = ["<|endoftext|>"]
vocab_size = 10000

vocab = {i: bytes([i]) for i in range(256)}
reverse_vocab = {token: idx for idx, token in vocab.items()}
merges = []

In [ ]:
from pathlib import Path

corpus_path = Path('/Users/andrewzabelo/Desktop/hw1/data/TinyStoriesV2-GPT4-train.txt')
dataset = corpus_path.read_text(encoding='utf-8')

pattern = "|".join(map(re.escape, special_tokens))
documents = re.split(pattern, dataset)

PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

In [36]:
print(len(documents))
pretoken_counts = {}
for document in documents:
    for match in re.finditer(PAT, document):
        pretoken = match.group(0)
        pretoken_counts[pretoken] = pretoken_counts.get(pretoken, 0) + 1

pretoken_linked_lists = {}
pair_counts = {}
pair_to_pretokens = {}

class ReverseLexPair:
    def __init__(self, left, right):
        self.left = left
        self.right = right

    def __lt__(self, other):
        return (self.left, self.right) > (other.left, other.right)

for pretoken in pretoken_counts:
    token_bytes = pretoken.encode('utf-8')
    token_ids = [reverse_vocab[bytes([b])] for b in token_bytes]
    for a, b in zip(token_ids, token_ids[1:]):
        pair_to_pretokens.setdefault((a, b), set())

# Four-array linked list: token ids, prev pointers, next pointers, alive flags.
for pretoken, count in pretoken_counts.items():
    token_bytes = pretoken.encode('utf-8')
    token_ids = [reverse_vocab[bytes([b])] for b in token_bytes]

    n = len(token_ids)
    prev = [i - 1 for i in range(n)]
    next_ = [i + 1 for i in range(n)]
    alive = [True] * n

    if n > 0:
        prev[0] = -1
        next_[-1] = -1

    pretoken_linked_lists[pretoken] = {
        'tokens': token_ids,
        'prev': prev,
        'next': next_,
        'alive': alive,
        'head': 0 if n > 0 else -1,
    }

    for a, b in zip(token_ids, token_ids[1:]):
        pair = (a, b)
        pair_counts[pair] = pair_counts.get(pair, 0) + count
        pair_to_pretokens[pair].add(pretoken)

def make_heap_entry(pair, count):
    return (-count, ReverseLexPair(vocab[pair[0]], vocab[pair[1]]), pair)

pair_counts_heap = [make_heap_entry(pair, count) for pair, count in pair_counts.items()]
heapq.heapify(pair_counts_heap)

2717700


In [38]:
print(pretoken_counts)

{'\n': 15011026, 'Once': 1712895, ' upon': 1628280, ' a': 15063529, ' time': 2158385, ' there': 2096034, ' was': 10593232, ' little': 2335215, ' boy': 1076463, ' named': 2064580, ' Ben': 800813, '.': 41764510, ' loved': 1049066, ' to': 14903559, ' explore': 92642, ' the': 20828576, ' world': 79126, ' around': 598251, ' him': 1134703, ' He': 4840028, ' saw': 2460060, ' many': 513998, ' amazing': 57580, ' things': 597863, ',': 23284330, ' like': 870753, ' beautiful': 203433, ' vases': 243, ' that': 2466207, ' were': 1995318, ' on': 2550422, ' display': 10257, ' in': 3848306, ' store': 157527, ' One': 887766, ' day': 4216755, ' walking': 123027, ' through': 88294, ' when': 572008, ' he': 3234511, ' came': 827901, ' across': 35011, ' very': 2509817, ' special': 282765, ' vase': 24545, ' When': 229115, ' it': 5138607, ' amazed': 40724, '!': 1883362, '  ': 1816, 'He': 128319, ' said': 4370380, ' “': 138863, 'Wow': 132066, ' is': 1685818, ' really': 65317, ' Can': 154379, ' I': 1859001, ' buy

In [40]:
max([len(pretoken) for pretoken in pretoken_counts.keys()])

1101

In [41]:
def remove_pair(pair, amount):
    if pair not in pair_counts:
        return

    new_count = pair_counts[pair] - amount
    if new_count > 0:
        pair_counts[pair] = new_count
        heapq.heappush(pair_counts_heap, make_heap_entry(pair, new_count))
    else:
        del pair_counts[pair]


def add_pair(pair, amount, pretoken):
    pair_counts[pair] = pair_counts.get(pair, 0) + amount
    if pair not in pair_to_pretokens:
        pair_to_pretokens[pair] = set()
    pair_to_pretokens[pair].add(pretoken)
    heapq.heappush(pair_counts_heap, make_heap_entry(pair, pair_counts[pair]))


while len(vocab) < vocab_size and pair_counts_heap:
    neg_count, _, pair = heapq.heappop(pair_counts_heap)
    if pair not in pair_counts or pair_counts[pair] != -neg_count:
        continue

    left_bytes = vocab[pair[0]]
    right_bytes = vocab[pair[1]]
    merged_bytes = left_bytes + right_bytes

    new_token_id = len(vocab)
    vocab[new_token_id] = merged_bytes
    reverse_vocab[merged_bytes] = new_token_id
    merges.append((left_bytes, right_bytes))

    affected_pretokens = list(pair_to_pretokens.get(pair, set()))

    for pretoken in affected_pretokens:
        linked_list = pretoken_linked_lists[pretoken]
        tokens = linked_list['tokens']
        prev = linked_list['prev']
        next_ = linked_list['next']
        alive = linked_list['alive']
        count = pretoken_counts[pretoken]

        node = linked_list['head']
        while node != -1:
            right = next_[node]
            if right == -1:
                break

            if tokens[node] != pair[0] or tokens[right] != pair[1]:
                node = right
                continue

            left_neighbor = prev[node]
            right_neighbor = next_[right]

            remove_pair(pair, count)

            if left_neighbor != -1:
                old_left_pair = (tokens[left_neighbor], tokens[node])
                remove_pair(old_left_pair, count)
                new_left_pair = (tokens[left_neighbor], new_token_id)
                add_pair(new_left_pair, count, pretoken)

            if right_neighbor != -1:
                old_right_pair = (tokens[right], tokens[right_neighbor])
                remove_pair(old_right_pair, count)
                new_right_pair = (new_token_id, tokens[right_neighbor])
                add_pair(new_right_pair, count, pretoken)

            tokens[node] = new_token_id
            next_[node] = right_neighbor
            if right_neighbor != -1:
                prev[right_neighbor] = node

            alive[right] = False
            prev[right] = -1
            next_[right] = -1

            node = next_[node]


In [42]:
print(vocab)

{0: b'\x00', 1: b'\x01', 2: b'\x02', 3: b'\x03', 4: b'\x04', 5: b'\x05', 6: b'\x06', 7: b'\x07', 8: b'\x08', 9: b'\t', 10: b'\n', 11: b'\x0b', 12: b'\x0c', 13: b'\r', 14: b'\x0e', 15: b'\x0f', 16: b'\x10', 17: b'\x11', 18: b'\x12', 19: b'\x13', 20: b'\x14', 21: b'\x15', 22: b'\x16', 23: b'\x17', 24: b'\x18', 25: b'\x19', 26: b'\x1a', 27: b'\x1b', 28: b'\x1c', 29: b'\x1d', 30: b'\x1e', 31: b'\x1f', 32: b' ', 33: b'!', 34: b'"', 35: b'#', 36: b'$', 37: b'%', 38: b'&', 39: b"'", 40: b'(', 41: b')', 42: b'*', 43: b'+', 44: b',', 45: b'-', 46: b'.', 47: b'/', 48: b'0', 49: b'1', 50: b'2', 51: b'3', 52: b'4', 53: b'5', 54: b'6', 55: b'7', 56: b'8', 57: b'9', 58: b':', 59: b';', 60: b'<', 61: b'=', 62: b'>', 63: b'?', 64: b'@', 65: b'A', 66: b'B', 67: b'C', 68: b'D', 69: b'E', 70: b'F', 71: b'G', 72: b'H', 73: b'I', 74: b'J', 75: b'K', 76: b'L', 77: b'M', 78: b'N', 79: b'O', 80: b'P', 81: b'Q', 82: b'R', 83: b'S', 84: b'T', 85: b'U', 86: b'V', 87: b'W', 88: b'X', 89: b'Y', 90: b'Z', 91: b'[',

In [43]:
import pickle
from pathlib import Path

vocab_output_path = Path('vocab.pkl')
with vocab_output_path.open('wb') as f:
    pickle.dump(vocab, f)

merges_output_path = Path('merges.pkl')
with merges_output_path.open('wb') as f:
    pickle.dump(merges, f)

special_tokens_output_path = Path('special_tokens.pkl')
with special_tokens_output_path.open('wb') as f:
    pickle.dump(special_tokens, f)

print(vocab_output_path.resolve())
print(merges_output_path.resolve())
print(special_tokens_output_path.resolve())


/Users/andrewzabelo/Desktop/hw1/tests/vocab.pkl
/Users/andrewzabelo/Desktop/hw1/tests/merges.pkl
/Users/andrewzabelo/Desktop/hw1/tests/special_tokens.pkl


In [44]:
vocab_input_path = Path('vocab.pkl')
with vocab_input_path.open('rb') as f:
    loaded_vocab = pickle.load(f)

merges_input_path = Path('merges.pkl')
with merges_input_path.open('rb') as f:
    loaded_merges = pickle.load(f)

special_tokens_input_path = Path('special_tokens.pkl')
with special_tokens_input_path.open('rb') as f:
    loaded_special_tokens = pickle.load(f)

In [45]:
longest_token_id, longest_token = max(loaded_vocab.items(), key=lambda item: len(item[1]))
print(longest_token_id)
print(longest_token)
print(len(longest_token))


7432
b' accomplishment'
15


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from eecs148b_hw1.tokenizer import Tokenizer

sample_documents = documents[:10]
tokenizer = Tokenizer.from_files('vocab.pkl', 'merges.pkl', special_tokens)

total_raw_bytes = 0
total_tokens = 0

for document in sample_documents:
    total_raw_bytes += len(document.encode('utf-8'))
    total_tokens += len(tokenizer.encode(document))

print(total_raw_bytes)
print(total_tokens)

print(total_raw_bytes / total_tokens)


NameError: name 'special_tokens' is not defined

In [8]:
import numpy as np

train_path = Path.cwd().parent / 'data' / 'TinyStoriesV2-GPT4-train.txt'
train_text = train_path.read_text(encoding='utf-8')
special_tokens = ["<|endoftext|>"]
tokenizer = Tokenizer.from_files('vocab.pkl', 'merges.pkl', special_tokens)
train_token_ids = np.array(tokenizer.encode(train_text), dtype=np.int16)

output_npy_path = Path.cwd().parent / 'data' / 'tinystories_train_tokens.npy'
np.save(output_npy_path, train_token_ids)

print(output_npy_path)
print(train_token_ids.shape)


/Users/andrewzabelo/Desktop/hw1/data/tinystories_train_tokens.npy
(545600227,)


In [7]:
valid_path = Path.cwd().parent / 'data' / 'TinyStoriesV2-GPT4-valid.txt'
valid_text = valid_path.read_text(encoding='utf-8')
special_tokens = ["<|endoftext|>"]
tokenizer = Tokenizer.from_files('vocab.pkl', 'merges.pkl', special_tokens)
valid_token_ids = np.array(tokenizer.encode(valid_text), dtype=np.int16)

output_npy_path = Path.cwd().parent / 'data' / 'tinystories_valid_tokens.npy'
np.save(output_npy_path, valid_token_ids)

print(output_npy_path)
print(valid_token_ids.shape)

/Users/andrewzabelo/Desktop/hw1/data/tinystories_valid_tokens.npy
(5501911,)
